# 09. PyTorch Model Deployment Exercises

Welcome to the 09. PyTorch Model Deployment exercises.

Your objective is to write code to satisify each of the exercises below.

Some starter code has been provided to make sure you have all the resources you need.

> **Note:** There may be more than one solution to each of the exercises.

## Resources

1. These exercises/solutions are based on [section 09. PyTorch Model Deployment](https://www.learnpytorch.io/09_pytorch_model_deployment/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.
2. See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/jOX5ZCkWO-0) (but try the exercises yourself first!).
3. See [all solutions on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/extras/solutions).

> **Note:** The first section of this notebook is dedicated to getting various helper functions and datasets used for the exercises. The exercises start at the heading "Exercise 1: ...".

In [1]:
import torch
import torchvision
# Get regular imports 
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision import transforms
from torchinfo import summary
from going_modular.going_modular.engine import train_test_step
from going_modular.going_modular.data_setup import create_dataloaders
from going_modular.going_modular.data_import import get_data
import os

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

## Exercise 3. Make predictions across the 20% Food101 test dataset with the ViT feature extractor from exercise 2 and find the "most wrong" predictions
* The predictions will be the ones with the highest prediction probability but with the wrong predicted label.
* Write a sentence or two about why you think the model got these predictions wrong.

In [3]:
from demos.foodvision_mini.model import create_vitb16_model
from going_modular.going_modular.utils import load_model
vitb16_food101, vitb16_transforms = create_vitb16_model(num_classes=101)
load_model(model_path="models/vitb16_food101_20_percent.pth", 
           model_builder=vitb16_food101, 
           device=device)


VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [4]:
summary(model=vitb16_food101, 
        input_size=(1, 3, 384, 384), 
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
VisionTransformer (VisionTransformer)                        [1, 3, 384, 384]     [1, 101]             768                  Partial
├─Conv2d (conv_proj)                                         [1, 3, 384, 384]     [1, 768, 24, 24]     (590,592)            False
├─Encoder (encoder)                                          [1, 577, 768]        [1, 577, 768]        443,136              False
│    └─Dropout (dropout)                                     [1, 577, 768]        [1, 577, 768]        --                   --
│    └─Sequential (layers)                                   [1, 577, 768]        [1, 577, 768]        --                   False
│    │    └─EncoderBlock (encoder_layer_0)                   [1, 577, 768]        [1, 577, 768]        (7,087,872)          False
│    │    └─EncoderBlock (encoder_layer_1)                   [1, 577, 768]        [1, 5

In [9]:
from pathlib import Path
test_dir = "data/food_101/food-101/test"
# Get all test data paths
print(f"[INFO] Finding all filepaths ending with '.jpg' in directory: {test_dir}")
test_data_paths = list(Path(test_dir).glob("*/*.jpg"))

# get class names
class_names = sorted([f.name for f in Path(test_dir).iterdir() if f.is_dir()])
print(f"[INFO] Found {len(class_names)} classes: {class_names}")

[INFO] Finding all filepaths ending with '.jpg' in directory: data/food_101/food-101/test
[INFO] Found 101 classes: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito', 'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake', 'ceviche', 'cheese_plate', 'cheesecake', 'chicken_curry', 'chicken_quesadilla', 'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros', 'clam_chowder', 'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame', 'cup_cakes', 'deviled_eggs', 'donuts', 'dumplings', 'edamame', 'eggs_benedict', 'escargots', 'falafel', 'filet_mignon', 'fish_and_chips', 'foie_gras', 'french_fries', 'french_onion_soup', 'french_toast', 'fried_calamari', 'fried_rice', 'frozen_yogurt', 'garlic_bread', 'gnocchi', 'greek_salad', 'grilled_cheese_sandwich', 'grilled_salmon', 'guacamole', 'gyoza', 'hamburger', 'hot_and_sour_soup', 'hot_dog', 'huevos_rancheros', 'humm

In [10]:
from going_modular.going_modular.utils import pred_and_store
vitb16_preds = pred_and_store(model=vitb16_food101,
                              paths=test_data_paths,
                              class_names=class_names,
                              transform=vitb16_transforms,
                              device=device)
vitb16_preds[:5]

  0%|          | 0/25250 [00:00<?, ?it/s]

[{'image_path': PosixPath('data/food_101/food-101/test/steak/3553838.jpg'),
  'class_names': 'steak',
  'pred_prob': '47.67 %',
  'pred_class': 'steak',
  'time_for_pred': 0.0276,
  'correct': True},
 {'image_path': PosixPath('data/food_101/food-101/test/steak/1882831.jpg'),
  'class_names': 'steak',
  'pred_prob': '63.21 %',
  'pred_class': 'filet_mignon',
  'time_for_pred': 0.0248,
  'correct': False},
 {'image_path': PosixPath('data/food_101/food-101/test/steak/2720938.jpg'),
  'class_names': 'steak',
  'pred_prob': '94.37 %',
  'pred_class': 'steak',
  'time_for_pred': 0.0176,
  'correct': True},
 {'image_path': PosixPath('data/food_101/food-101/test/steak/928920.jpg'),
  'class_names': 'steak',
  'pred_prob': '85.85 %',
  'pred_class': 'filet_mignon',
  'time_for_pred': 0.0152,
  'correct': False},
 {'image_path': PosixPath('data/food_101/food-101/test/steak/429304.jpg'),
  'class_names': 'steak',
  'pred_prob': '41.17 %',
  'pred_class': 'prime_rib',
  'time_for_pred': 0.0177,
  

In [11]:
import pandas as pd


pred_prob_df = pd.DataFrame(vitb16_preds)
pred_prob_df.head()

,image_path,class_names,pred_prob,pred_class,time_for_pred,correct
0,data/food_101/food-101/test/steak/3553838.jpg,steak,47.67 %,steak,0.0276,True
1,data/food_101/food-101/test/steak/1882831.jpg,steak,63.21 %,filet_mignon,0.0248,False
2,data/food_101/food-101/test/steak/2720938.jpg,steak,94.37 %,steak,0.0176,True
3,data/food_101/food-101/test/steak/928920.jpg,steak,85.85 %,filet_mignon,0.0152,False
4,data/food_101/food-101/test/steak/429304.jpg,steak,41.17 %,prime_rib,0.0177,False


In [29]:
pred_prob_df_wrong = pred_prob_df[pred_prob_df["correct"] == False]
pred_prob_df_wrong.head()

,image_path,class_names,pred_prob,pred_class,time_for_pred,correct
1,data/food_101/food-101/test/steak/1882831.jpg,steak,63.21 %,filet_mignon,0.0248,False
3,data/food_101/food-101/test/steak/928920.jpg,steak,85.85 %,filet_mignon,0.0152,False
4,data/food_101/food-101/test/steak/429304.jpg,steak,41.17 %,prime_rib,0.0177,False
5,data/food_101/food-101/test/steak/2984679.jpg,steak,43.52 %,filet_mignon,0.0180,False
6,data/food_101/food-101/test/steak/66183.jpg,steak,86.38 %,filet_mignon,0.0158,False


In [30]:
pred_prob_df_wrong = pred_prob_df_wrong.copy()
pred_prob_df_wrong["pred_prob"] = pred_prob_df_wrong["pred_prob"].str.replace("%", "").astype("float")
pred_prob_df_wrong

,image_path,class_names,pred_prob,pred_class,time_for_pred,correct
1,data/food_101/food-101/test/steak/1882831.jpg,steak,63.21,filet_mignon,0.0248,False
3,data/food_101/food-101/test/steak/928920.jpg,steak,85.85,filet_mignon,0.0152,False
4,data/food_101/food-101/test/steak/429304.jpg,steak,41.17,prime_rib,0.0177,False
5,data/food_101/food-101/test/steak/2984679.jpg,steak,43.52,filet_mignon,0.0180,False
6,data/food_101/food-101/test/steak/66183.jpg,steak,86.38,filet_mignon,0.0158,False
...,...,...,...,...,...,...
25212,data/food_101/food-101/test/takoyaki/185141.jpg,takoyaki,28.03,beet_salad,0.0172,False
25224,data/food_101/food-101/test/takoyaki/3332604.jpg,takoyaki,39.95,donuts,0.0197,False
25226,data/food_101/food-101/test/takoyaki/3153904.jpg,takoyaki,18.64,caesar_salad,0.0160,False
25236,data/food_101/food-101/test/takoyaki/2189202.jpg,takoyaki,53.59,beet_salad,0.0190,False


In [36]:
pred_prob_df_wrong_sorted = pred_prob_df_wrong.sort_values("pred_prob", ascending=False)
pred_prob_df_wrong_sorted.head()

,image_path,class_names,pred_prob,pred_class,time_for_pred,correct
16316,data/food_101/food-101/test/beef_tartare/29563...,beef_tartare,99.57,beet_salad,0.0191,False
16241,data/food_101/food-101/test/gyoza/2741597.jpg,gyoza,99.46,dumplings,0.0150,False
16777,data/food_101/food-101/test/scallops/3557461.jpg,scallops,99.37,shrimp_and_grits,0.0182,False
17463,data/food_101/food-101/test/risotto/547531.jpg,risotto,99.30,macaroni_and_cheese,0.0183,False
7274,data/food_101/food-101/test/cup_cakes/137287.jpg,cup_cakes,99.22,chocolate_cake,0.0153,False


## Exercise 7. Pick any dataset from [`torchvision.datasets`](https://pytorch.org/vision/stable/datasets.html) and train a feature extractor model on it using a model from [`torchvision.models`](https://pytorch.org/vision/stable/models.html) (you could use one of the model's we've already created, e.g. EffNetB2 or ViT) for 5 epochs and then deploy your model as a Gradio app to Hugging Face Spaces. 
* You may want to pick smaller dataset/make a smaller split of it so training doesn't take too long.
* I'd love to see your deployed models! So be sure to share them in Discord or on the [course GitHub Discussions page](https://github.com/mrdbourke/pytorch-deep-learning/discussions).